# Day 7 · Titanic 第一轮 EDA(W1 收官)

目标:走一遍完整 EDA 五步:**读数据 → 概览 → 清洗 → 单变量 → 双变量 → 结论**。

数据:仓库里的 200 行样本 `../datasets/titanic-sample.csv`(与 Kaggle `train.csv` 同结构,可整文件替换成完整版)。

今日节奏:40min 学习 + 15min 动手 + 5min 自检。

> 打开方式:JupyterLab 文件树里进 `ai-learning/练习/` 双击本文件,逐格 Shift+Enter。


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)

titanic = pd.read_csv(os.path.join("..", "datasets", "titanic-sample.csv"))
print("形状:", titanic.shape)
print(titanic.head())
print(titanic.info())


## 任务 1:数据概览(先体检,再分析)

任何 EDA 的第一分钟都花在这四件事上:

- `.isna().sum()` → 每列缺多少
- `.duplicated().sum()` → 有没有重复行
- `.value_counts()` → 类别列的取值分布
- `.describe()` → 数值列分布概况


In [ ]:
print("每列缺失:\n", titanic.isna().sum())
print("重复行:", titanic.duplicated().sum())
print("存活分布:\n", titanic["Survived"].value_counts())
print("Pclass 分布:", titanic["Pclass"].value_counts().sort_index().to_dict())
print(titanic.describe().round(1))


## 任务 2:清洗缺失值

策略要对症下药:
- `Age`(数值)→ 用**中位数**补(抗极端值)
- `Embarked`(类别)→ 用**众数**补(最常见的取值)
- 补完必须复查:`isna().sum()` 应为全 0


In [ ]:
fixed = titanic.copy()
fixed["Age"] = fixed["Age"].fillna(fixed["Age"].median())
fixed["Embarked"] = fixed["Embarked"].fillna(fixed["Embarked"].mode()[0])
print("清洗后缺失:\n", fixed.isna().sum())


## 任务 3:单变量分析(总存活率 + 分组存活率)

先看总体存活率,再按两个候选特征拆开:
- 性别:女性存活率是不是明显更高?
- 舱位(Pclass):一等舱是不是更容易活下来?


In [ ]:
print("总存活率:", round(fixed["Survived"].mean() * 100, 1), "%")
print("按性别:\n", fixed.groupby("Sex")["Survived"].mean().round(3))
print("按舱位:\n", fixed.groupby("Pclass")["Survived"].mean().round(3))

plt.figure(figsize=(8, 3))

plt.subplot(1, 2, 1)
fixed.groupby("Sex")["Survived"].mean().plot(kind="bar", color=["#d98880", "#5dade2"], rot=0)
plt.ylabel("Survival rate")
plt.title("Survival by sex")

plt.subplot(1, 2, 2)
fixed.groupby("Pclass")["Survived"].mean().plot(kind="bar", color="steelblue", rot=0)
plt.ylabel("Survival rate")
plt.title("Survival by class")

plt.tight_layout()
plt.show()


## 任务 4:双变量分析(交叉看)

- 性别 × 舱位 的存活率矩阵:`pivot_table`
- 年龄分布按存活/遇难分两组画直方图
- 票价按舱位画箱线图(boxplot:中位数、四分位、离群点)


In [ ]:
pivot = fixed.pivot_table(index="Sex", columns="Pclass", values="Survived", aggfunc="mean").round(2)
print("性别×舱位存活率:\n", pivot)

plt.figure(figsize=(8, 3))

plt.subplot(1, 2, 1)
fixed[fixed["Survived"] == 1]["Age"].hist(bins=15, alpha=0.7, color="green", label="Survived")
fixed[fixed["Survived"] == 0]["Age"].hist(bins=15, alpha=0.6, color="red", label="Not survived")
plt.xlabel("Age")
plt.ylabel("Count")
plt.legend()
plt.title("Age by survival")

plt.subplot(1, 2, 2)
fixed.boxplot(column="Fare", by="Pclass", grid=False)
plt.suptitle("")
plt.xlabel("Pclass")
plt.ylabel("Fare")
plt.title("Fare by class")

plt.tight_layout()
plt.show()


## 任务 5:小挑战 🔥(找到"生存密码")

1. 按 性别×舱位 分组,找出存活率最高和最低的组合
2. 算各数值列与 Survived 的相关系数,哪个特征最"有用"?


In [ ]:
combo = fixed.groupby(["Sex", "Pclass"])["Survived"].agg(["mean", "count"]).round(3)
print("分组存活率(从高到低):\n", combo.sort_values("mean", ascending=False))

corr = fixed[["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"]].corr()["Survived"].round(3)
print("与存活的相关性:\n", corr)


## 自检清单(5 问,答不上就回看今天的格子)

1. EDA 是什么? → 探索性数据分析:读→概览→清洗→单变量→双变量→结论
2. 为什么 Age 用中位数补? → 中位数抗极端值,比均值稳
3. pivot_table 干了什么? → 行×列的交叉聚合(性别×舱位的存活率)
4. boxplot 能看什么? → 中位数、四分位、离群点
5. 今天最强的三个"特征"? → Sex、Pclass、Age(Fare 其次)

## 📝 收盘动作

```powershell
cd D:\01_Study\ai-learning
git add -A; git commit -m "day7: titanic eda"; git push
```

然后跟助手说"生成日志"。
